In [3]:
import os
from sys import path

import torch
import json
from pprint import pprint
from PIL import Image

In [8]:
root=r"C:\Users\BAOHUY\Downloads\TACO dataset.v1i.coco\train"
with open(os.path.join(root, "_annotations.coco.json")) as f:
    coco = json.load(f)

# # Sau khi load xong, thực hành truy cập
#
# # Xem tất cả ảnh
# for img in coco["images"]:
#     print(f"Ảnh ID {img['id']}: {img['file_name']} ({img['width']}x{img['height']})")
#
# # Xem tất cả nhãn
for cat in coco["categories"]:
    print(f"Nhãn ID {cat['id']}: {cat['name']}")
#
# # Xem annotations
# for ann in coco["annotations"]:
#     print(f"Annotation {ann['id']}: ảnh={ann['image_id']}, nhãn={ann['category_id']}, bbox={ann['bbox']}")

Nhãn ID 0: trash
Nhãn ID 1: cardboard
Nhãn ID 2: glass
Nhãn ID 3: metal
Nhãn ID 4: other
Nhãn ID 5: paper
Nhãn ID 6: plastic


In [13]:
print(coco.keys())
pprint(coco["images"][40])
pprint(coco["annotations"][80])

img_to_anns = {}
for ann in coco["annotations"]:
    img_id = ann["image_id"]
    img_to_anns.setdefault(img_id, []).append(ann)
pprint(img_to_anns)

dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])
{'date_captured': '2026-04-27T09:38:56+00:00',
 'extra': {'name': '000015.jpg'},
 'file_name': '000015_jpg.rf.dffd908b7188dccc758ea9daa801dfc5.jpg',
 'height': 640,
 'id': 40,
 'license': 1,
 'width': 640}
{'area': 22663.5,
 'bbox': [262, 327, 260.5, 87],
 'category_id': 6,
 'id': 81,
 'image_id': 32,
 'iscrowd': 0,
 'segmentation': []}
{0: [{'area': 58292.75,
      'bbox': [355, 36, 270.5, 215.5],
      'category_id': 6,
      'id': 1,
      'image_id': 0,
      'iscrowd': 0,
      'segmentation': []},
     {'area': 10339.5,
      'bbox': [130, 547, 169.5, 61],
      'category_id': 5,
      'id': 2,
      'image_id': 0,
      'iscrowd': 0,
      'segmentation': []}],
 1: [{'area': 1456,
      'bbox': [336, 327, 56, 26],
      'category_id': 6,
      'id': 3,
      'image_id': 1,
      'iscrowd': 0,
      'segmentation': []}],
 2: [{'area': 324233,
      'bbox': [0, 71, 637, 509],
      'category_id': 6,
      'id':

In [6]:
image = coco["images"][1]["file_name"]
image_path = os.path.join(root, image)
image = Image.open(image_path).convert("RGB")
print(image.size)
print(image,image_path)

(640, 640)
<PIL.Image.Image image mode=RGB size=640x640 at 0x19D27C47690> C:\Users\BAOHUY\Downloads\TACO dataset.v1i.coco\train\000025_JPG.rf.7a9cb5f7ce00f7b0c380819f365aad31.jpg


In [1]:
import torch
import os
import json
from PIL import Image
class TrashDataset(torch.utils.data.Dataset):
    def __init__(self, root, split="train",transforms=None):
        self.root = os.path.join(root, split)
        self.transforms = transforms

        with open(os.path.join(self.root, "_annotations.coco.json")) as f:
            coco = json.load(f)

        self.images = coco["images"]
        self.annotations = coco["annotations"]

        """
            10: [{'area': 5687,
           'bbox': [198, 207, 47, 121],
           'category_id': 6,
           'id': 16,
           'image_id': 10,
           'iscrowd': 0,
           'segmentation': []},
          {'area': 30342.5,
           'bbox': [275, 108, 114.5, 265],
           'category_id': 6,
           'id': 17,
           'image_id': 10,
           'iscrowd': 0,
           'segmentation': []},
          {'area': 29550,
           'bbox': [104, 337, 100, 295.5],
           'category_id': 6,
           'id': 18,
           'image_id': 10,
           'iscrowd': 0,
           'segmentation': []}],
        """
        self.img_to_anns = {}
        for ann in self.annotations:
            self.img_to_anns.setdefault(ann["image_id"],[]).append(ann)

    def __getitem__(self, item):
        image_name = self.images[item]['file_name']
        imgae_id = self.images[item]['id']

        image_path = os.path.join(self.root, image_name)
        image = Image.open(image_path).convert('RGB')

        anns = self.img_to_anns.get(imgae_id,[])
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x+w, y+h])
            labels.append(ann['category_id'])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

    def __len__(self):
        return len(self.images)





In [2]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = TrashDataset(
    root=r"C:\Users\BAOHUY\Downloads\TACO dataset.v1i.coco",
    split="train"
)

data_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

for imgs, targets in data_loader:
    print(len(imgs))  # batch size
    print(targets[0])
    break

2
{'boxes': tensor([[  0.0000,  36.0000, 474.0000, 434.0000],
        [295.0000, 406.0000, 448.5000, 532.0000]]), 'labels': tensor([6, 6])}


In [1]:
import json
from pprint import pprint
with open(r"C:\Users\BAOHUY\Downloads\TACO dataset.v1i.coco\train\_annotations.coco.json") as f:
    coco = json.load(f)

pprint(coco["categories"])
max_id = max(ann["category_id"] for ann in coco["annotations"])
print(f"Max category_id: {max_id}")
print(f"→ num_classes nên là: {max_id + 1}")

[{'id': 0, 'name': 'trash', 'supercategory': 'none'},
 {'id': 1, 'name': 'cardboard', 'supercategory': 'trash'},
 {'id': 2, 'name': 'glass', 'supercategory': 'trash'},
 {'id': 3, 'name': 'metal', 'supercategory': 'trash'},
 {'id': 4, 'name': 'other', 'supercategory': 'trash'},
 {'id': 5, 'name': 'paper', 'supercategory': 'trash'},
 {'id': 6, 'name': 'plastic', 'supercategory': 'trash'}]
Max category_id: 6
→ num_classes nên là: 7


In [4]:
img, tar = dataset[80]

print(img)
print(tar)

<PIL.Image.Image image mode=RGB size=640x640 at 0x26016A87810>
{'boxes': tensor([[177.0000, 461.0000, 510.0000, 602.0000],
        [ 80.0000, 138.0000, 418.5000, 239.5000]]), 'labels': tensor([6, 6])}
